# FPN - Uydudan bina hasari segmentasyonu

**Katman:** Zorunlu  |  `resnet18` / `FPN`

Ozellik piramidi; M2'de yuksek dogruluk/ozgulluk.

Yol modulu (`01_yol_hasari`) ile birebir ayni protokol -> M2 ile karsilastirilabilir.  
Metrik: IoU/mIoU, F1, Kesinlik, Duyarlilik, Ozgulluk, Dogruluk, AUC + Params/GFLOPs/cikarim.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/02_bina_hasari
    %pip install -q segmentation-models-pytorch albumentations roboflow thop opencv-python-headless
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import seg_common as SC
SC.CONFIG.update(dict(
    source='roboflow',
    roboflow_api_key='BURAYA_ROBOFLOW_API_KEY',
    roboflow_workspace='workspace-slug',
    roboflow_project='project-slug',
    roboflow_version=1,
    num_classes=1,   # xBD 4 hasar sinifi kullanilacaksa 4
))
paths = SC.prepare_dataset(); dl = SC.build_loaders(paths)
print({k: len(v.dataset) for k, v in dl.items()})

## 2. Model

In [ ]:
MODEL_NAME = "fpn_resnet18"
ARCH, ENCODER = "FPN", "resnet18"
PARAMS = dict(arch=ARCH, encoder=ENCODER, **{k: SC.CONFIG[k] for k in ['img_size','batch_size','epochs','lr']})
model = SC.build_model(ARCH, ENCODER, SC.CONFIG['num_classes'])

## 3. Egitim

In [ ]:
model = SC.train_model(model, dl, SC.CONFIG)

## 4. Test + verimlilik

In [ ]:
import torch; dev = 'cuda' if torch.cuda.is_available() else 'cpu'
split = 'test' if 'test' in dl else 'valid'
metrics = SC.evaluate_seg(model, dl[split], dev)
eff = SC.efficiency_stats(model, SC.CONFIG['img_size'], 'cpu')
for k, v in metrics.items(): print(f'{k:12s}: {v:.4f}')
print(eff)

## 5. Kaydet

In [ ]:
lb = SC.save_result(MODEL_NAME, metrics, eff, extra={'params': PARAMS}); lb